In [ ]:
%config Completer.use_jedi = False

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import RobustScaler, MinMaxScaler, StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

import matplotlib.pyplot as plt

In [ ]:
def preprocess_data(df):
    df.loc[df['height'] < 120, 'height'] = np.nan
    df['height'] = df['height'].fillna(df['height'].mean())
    
    df.loc[(df['weight'] >= 140)|(df['weight'] < 30), 'weight'] = np.nan
    df['weight'] = df['weight'].fillna(df['weight'].mean())
    
    df.loc[df['respiratoryrate_value'] >= 40, 'respiratoryrate_value'] = np.nan
    df['respiratoryrate_value'] = df['respiratoryrate_value'].fillna(df['respiratoryrate_value'].mean())
    
    df.loc[df['gcs_value'] > 15, 'gcs_value'] = np.nan
    df['gcs_value'] = df['gcs_value'].fillna(df['gcs_value'].mean())
    
    df.loc[df['heartrate_value'] > 150, 'heartrate_value'] = np.nan
    df['heartrate_value'] = df['heartrate_value'].fillna(df['heartrate_value'].mean())
    
    df.loc[(df['dbp_value'] < 20) | (df['dbp_value'] >= 150), 'dbp_value'] = np.nan
    df['dbp_value'] = df['dbp_value'].fillna(df['dbp_value'].mean())
    
    df.loc[(df['sbp_value'] < 50) | (df['sbp_value'] >= 250), 'sbp_value'] = np.nan
    df['sbp_value'] = df['sbp_value'].fillna(df['sbp_value'].mean())
    
    df.loc[(df['map_value'] < 30) | (df['map_value'] >= 140), 'map_value'] = np.nan
    df['map_value'] = df['map_value'].fillna(df['map_value'].mean())
    
    df.loc[df['spo_value'] < 85, 'spo_value'] = np.nan
    df['spo_value'] = df['spo_value'].fillna(df['spo_value'].mean())
    
    df.loc[(df['bt_value'] < 35) | (df['bt_value'] > 40), 'bt_value'] = np.nan
    df['bt_value'] = df['bt_value'].fillna(df['bt_value'].mean())
    
    df.loc[df['pco_value'] >= 100, 'pco_value'] = np.nan
    df['pco_value'] = df['pco_value'].fillna(df['pco_value'].mean())
    
    df['BMI'] = df['weight'] / ((df['height'] / 100) ** 2)
    
    df['PF_ratio'] = df['pao_value'] / df['fio_value']
    
    comorbidities = ['TBI', 'SAH', 'ICH', 'IS', 'cardiac', 'resp', 'neuro', 'dm', 'cd']
    df['comorbidities_count'] = df[comorbidities].sum(axis=1)
    df['comorbidities']= (df['comorbidities_count']<3).astype(int)

    y1_comorbidities1_count = len(df[(df['y_extubation'] == 1) & (df['comorbidities_count'] <3)])
    y0_comorbidities0_count = len(df[(df['y_extubation'] == 0) & (df['comorbidities_count'] <3)])
    print("y_extubation=1 and comorbidities<3: ", y1_comorbidities1_count, " of ",len(df[df['y_extubation']==1]))
    print("y_extubation=0 and comorbidities<3: ", y0_comorbidities0_count, " of ",len(df[df['y_extubation']==0]))
    #df=df.drop(columns=['comorbidities_count'])


    ### 의식 수준 평가
    df['awake_count']=((df['gcs_value']>=10) & (df['rass_value']<=-1)).astype(int)    
    y1_awake_count = len(df[(df['y_extubation'] == 1) & (df['awake_count'] ==1)])
    y0_awake_count = len(df[(df['y_extubation'] == 0) & (df['awake_count'] ==1)])
    print("y_extubation=1 & awake: ", y1_awake_count, " of ",len(df[df['y_extubation']==1]))
    print("y_extubation=0 & awake: ", y0_awake_count, " of ",len(df[df['y_extubation']==0]))
    
    df['tidal_volume'] = ((df['tidalvolumeobserved_value']/df['weight']))
    df['tidal'] = (df['tidal_volume']>=4).astype(int)

    return df

In [ ]:
from imblearn.over_sampling import SMOTE
data = pd.read_csv("Train dataset.csv")
drop_cols = ['subject_id','extu_d']
data = data.drop(drop_cols,axis=1)

data=preprocess_data(data)

X = data.drop(columns=['y_extubation'])
y = data['y_extubation']

print((X.columns))

drop_cols_best= ['TBI', 'SAH', 'ICH', 'IS', 'cardiac', 'resp', 'neuro', 'dm', 'cd','fio_value','albuminlab_value'] ### best
drop_colss = ['TBI', 'SAH', 'ICH', 'IS', 'cardiac', 'neuro', 'dm', 'cd','resp','fio_value']
drop_colz = ['TBI', 'SAH', 'ICH', 'IS', 'cardiac', 'resp', 'neuro', 'dm', 'cd','fio_value','albuminlab_value','PEEPset_value','tidalvolumeobserved_value','ventilatortime_value','sbt_value']
X = X.drop(columns=drop_colss)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

smote = SMOTE(random_state=100)
X_train, y_train = smote.fit_resample(X_train, y_train)

scaler1 = RobustScaler()
X_train = scaler1.fit_transform(X_train)
X_test = scaler1.transform(X_test)
# scaler2 = RobustScaler()
# X_train = scaler2.fit_transform(X_train)
# X_test = scaler2.transform(X_test)


In [ ]:
def main(train_df, test_df):
    drop_cols = ['subject_id', 'extu_d']
    train_df = train_df.drop(drop_cols, axis = 1)
    test_df = test_df.drop(drop_cols, axis = 1)
    test_df = test_df[train_df.columns]

    train_df = preprocess_data(train_df)
    test_df = preprocess_data(test_df)

    drop_closs = ['TBI', 'SAH', 'ICH', 'IS', 'cardiac', 'resp', 'neuro', 'dm', 'cd', 'albuminlab_value']
    train_df = train_df.drop(columns = drop_colss)
    test_df = test_df.drop(columns=drop_colss)

    c = list(train_df.drop(['y_extubation'], axis=1).columns)
    c.append('y_extubation')

    X_test = test_df.drop(['y_extubation'], axis=1)
    y_test = test_df['y_extubation']

    X_train = train_df.drop(['y_extubation'], axis=1)
    y_train = train_df['y_extubation']

    smote = SMOTE(random_state = 42)
    X_train, y_train = smote.fit_resample(X_train, y_train)

    scaler = RobustScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    warnings.filterwarnings('ignore')

    clf = TabNetClassifier(
        n_step = 3,
        cat_idxs = [],
        cat_dims = [],
        cat_emb_dim = 2,
        optimizer_fn = torch.optim.Adam,
        optimizer_params = {"lr": 2e-2},
        scheduler_fn = None,
        clip_value = 2.0,
        momentum = 0.3,
        scheduler_params = {"step_size": 50, "gamma": 0.9},
        mask_type = 'sparasemax',
        device_name = 'cuda',
        n_d = 8,
        n_a = 8,
        seed = 21,
        lambda_sparse = 1e-3,
        epsilon = 1e-15
    )

    clf.fit(
    X_train, y_train,
    eval_set = [(X_train, y_train), (X_test, y_test)],
    eval_name = ['train', 'val'],
    eval_metric = ['auc', 'logloss'],
    max_epochs = 100,
    patience = 10,
    batch_size = 2,
    virtual_batch_size = 128,
    num_workers = 0,
    drop_last = False
    )

    train_df = pd.concat([pd.DataFrame(X_train), y_train], axis = 1)
    test_df = pd.concat([pd.DataFrame(X_test), y_test], axis = 1)
    train_df.columns = c
    test_df.columns = c
    model = clf

    return model, train_df, test_df

In [ ]:
from pytorch_tabnet.tab_model import TabNetClassifier
import torch

clf = TabNetClassifier(
    n_steps=3,
    cat_idxs=[],
    cat_dims=[],
    cat_emb_dim=64,
    optimizer_fn=torch.optim.Adam,
    optimizer_params={"lr": 29e-3},  
    scheduler_fn=None,
    clip_value=2.0,
    momentum=0.3,
    scheduler_params={"step_size": 50, "gamma": 0.9},  # 스케줄러 파라미터 수정
    mask_type='sparsemax',
    device_name='cuda',
    n_d=16,
    n_a=8,
    seed=100,
    lambda_sparse=0.3,  # 정규화 강도 증가
    epsilon=1e-15,
)

clf.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_test, y_test)],
    eval_name=['train', 'val'],
    eval_metric=['auc', 'logloss'], 
    max_epochs=1000,
    patience=200,  # 조기 종료를 위한 patience 설정
    batch_size=32,
    virtual_batch_size=128,
    num_workers=0,
    drop_last=False
)


In [ ]:
plt.plot(clf.history['loss'],marker='o', label='train')
plt.plot(clf.history['val_logloss'],marker='x', label='val')
plt.title('Loss per epoch')
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend()
plt.show()

In [ ]:
plt.plot(clf.history['train_auc'],marker='o', label='train')
plt.plot(clf.history['val_auc'],marker='x', label='val')
plt.title('acc per epoch')
plt.ylabel('acc')
plt.xlabel('epoch')
plt.legend()
plt.show()


In [ ]:
best_epoch = np.argmax(clf.history['val_auc'])
best_val_auc = clf.history['val_auc'][best_epoch]
best_train_auc = clf.history['train_auc'][best_epoch]

print(f"Best Epoch: {best_epoch+1}, Best train AUC: {best_train_auc:.4f}, Best Val AUC: {best_val_auc:.4f}")